# Preparación de datos — RSNA Pneumonia (prueba en Colab)

In [1]:
# Bloque 0.1 — Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Bloque 0.2 — Instalar y importar
!pip install -q pydicom

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

In [ ]:
# Bloque 0.3 — Rutas y búsqueda del ZIP
# Colab monta el Drive como 'MyDrive' (aunque tu cuenta esté en español).
base_drive = Path('/content/drive')
mi_unidad = base_drive / 'MyDrive'

# Carpeta DATASETS: Mi unidad > TESIS > DATASETS
# El ZIP se busca directo en DATASETS y en su subcarpeta 'zip'.
CARPETA_DATASETS = mi_unidad / 'TESIS' / 'DATASETS'
zips = sorted(CARPETA_DATASETS.glob('*.zip'))
print('ZIP encontrados:', [z.name for z in zips])
assert len(zips) > 0, 'No hay ningún .zip en TESIS > DATASETS'

ZIP encontrados: ['rsna-pneumonia-detection-challenge.zip']


In [7]:
# Bloque 0.4 — Descomprimir (tarda unos minutos, solo la primera vez)
for z in zips:
    destino = CARPETA_DATASETS / z.stem  # carpeta con el nombre del zip
    if (destino / 'stage_2_train_labels.csv').exists():
        print(f'Ya descomprimido: {destino.name} (se reutiliza)')
    else:
        print(f'Descomprimiendo {z.name} ... (tarda unos minutos)')
        destino.mkdir(parents=True, exist_ok=True)
        !unzip -q '{z}' -d '{destino}'
        print('Listo.')

Descomprimiendo rsna-pneumonia-detection-challenge.zip ... (tarda unos minutos)
Listo.


In [8]:
# Bloque 0.5 — Ubicar el dataset original
# Encuentra la carpeta con los CSV (por si el ZIP traía subcarpetas).
# Aquí solo se definen rutas a lo ORIGINAL (CSV + DICOMs); lo que este
# notebook genera se define en el bloque que lo usa (1.3, 1.4, 3, 4 y 5).
CARPETA_DATASET = None  # aquí quedará la carpeta con los CSV + DICOMs
for z in zips:
    destino = CARPETA_DATASETS / z.stem
    candidatos = [p.parent for p in destino.rglob('stage_2_train_labels.csv')]
    assert len(candidatos) > 0, f'No se halló stage_2_train_labels.csv dentro de {z.name}'
    CARPETA_DATASET = candidatos[0]

print('Carpeta del dataset:', CARPETA_DATASET)

# Rutas a los archivos ORIGINALES (iguales que en preparacion_datos.py)
CSV_TRAIN_LABELS = CARPETA_DATASET / 'stage_2_train_labels.csv'
CSV_CLASS_INFO = CARPETA_DATASET / 'stage_2_detailed_class_info.csv'
CARPETA_IMAGENES_TRAIN = CARPETA_DATASET / 'stage_2_train_images'


Carpeta del dataset: /content/drive/MyDrive/TESIS/DATASETS/rsna-pneumonia-detection-challenge


## Bloque 1 — Fase 1: Filtrado (selección de registros)

Deja 1 fila por imagen, se queda solo con `Lung Opacity` + `Normal` y filtra adultos de 18–100 años. Guarda la lista maestra en `dataset_filtrado.csv`.

In [9]:
# Bloque 1.1 — Cargar, deduplicar, cruzar y validar coherencia
print('=' * 70)
print('FASE 1 — FILTRADO (selección de registros)')
print('=' * 70)

# 1.1 Cargar los dos CSV originales
df_train_labels = pd.read_csv(CSV_TRAIN_LABELS)
df_class_info = pd.read_csv(CSV_CLASS_INFO)
print(f'\n#1. Filas leídas: train_labels={len(df_train_labels):,} | class_info={len(df_class_info):,}')

# 1.2 Deduplicar: 1 fila por imagen.
# Si la imagen tiene al menos una marca de neumonía -> Target=1
df_target = df_train_labels.groupby('patientId', as_index=False)['Target'].max()
df_clase = df_class_info.drop_duplicates('patientId')
print(f'#2. Imágenes únicas tras deduplicar: {len(df_target):,}')

# 1.3 Cruzar ambos CSV por patientId (deben calzar 1 a 1, sin pérdidas)
df = df_target.merge(df_clase, on='patientId', how='inner')
assert len(df) == len(df_target), 'Al cruzar se perdieron imágenes, revisar'
print(f'#3. Cruce por patientId OK: {len(df):,} imágenes (no se perdió ninguna)')

# 1.4 Validar coherencia: Target=1 debe ser class=Lung Opacity y viceversa.
# Chequeo de seguridad: si algún registro no concuerda, los CSV no calzan
# y el código se detiene antes de seguir filtrando con datos mezclados.
incoherentes = df[((df['Target'] == 1) != (df['class'] == 'Lung Opacity'))]
print(f'#4. Registros incoherentes Target vs class: {len(incoherentes):,}')
assert len(incoherentes) == 0, 'Hay incoherencias entre Target y class'

FASE 1 — FILTRADO (selección de registros)

#1. Filas leídas: train_labels=30,227 | class_info=30,227
#2. Imágenes únicas tras deduplicar: 26,684
#3. Cruce por patientId OK: 26,684 imágenes (no se perdió ninguna)
#4. Registros incoherentes Target vs class: 0


In [10]:
# Bloque 1.2 — Selección de clases: solo Lung Opacity + Normal.
# Se excluye No Lung Opacity/Not Normal (otra patología, no sirve al binario).
df = df[df['class'].isin(['Lung Opacity', 'Normal'])].copy()
df['etiqueta'] = (df['class'] == 'Lung Opacity').astype(int)  # 1=sí, 0=no
print('#5. Solo Lung Opacity + Normal (se excluye No Lung Opacity/Not Normal):')
for clase, conteo in df['class'].value_counts().items():
    print(f'   - {clase}: {conteo:,}')

#5. Solo Lung Opacity + Normal (se excluye No Lung Opacity/Not Normal):
   - Normal: 8,851
   - Lung Opacity: 6,012


In [11]:
# Bloque 1.3 — Filtro de edad: solo adultos 18-100 (criterio de inclusión de la tesis).
# Fuera pediátricos (<18), atípicos (>100, error de registro) y sin edad.
# La edad se lee del header de cada DICOM una sola vez y se guarda en caché.
EDAD_MINIMA = 18  # criterio de inclusión: solo adultos
EDAD_MAXIMA = 100  # mayor a 100 se considera error de registro (atípico)
# Caché de headers: se guarda en DATASETS para no re-leer los 26,684 DICOMs
CSV_HEADERS = CARPETA_DATASETS / 'dicom_headers.csv'

if CSV_HEADERS.exists():
    df_headers = pd.read_csv(CSV_HEADERS)[['patientId', 'edad', 'sexo', 'vista']]
    print('   - (headers cargados desde caché dicom_headers.csv)')
else:
    print('   - Leyendo headers de los DICOMs (tarda unos minutos, solo la primera vez)...')
    import pydicom
    filas_header = []
    for dcm in sorted(CARPETA_IMAGENES_TRAIN.glob('*.dcm')):
        ds = pydicom.dcmread(str(dcm), stop_before_pixels=True)
        edad_txt = str(ds.get('PatientAge', '')).strip()
        digitos = ''.join(ch for ch in edad_txt if ch.isdigit())
        filas_header.append({
            'patientId': dcm.stem,
            'edad': int(digitos) if digitos else None,
            'sexo': str(ds.get('PatientSex', '')).strip() or 'VACIO',
            'vista': str(ds.get('ViewPosition', '')).strip() or 'VACIO',
        })
    df_headers = pd.DataFrame(filas_header)
    df_headers.to_csv(CSV_HEADERS, index=False)
    print('   - (headers guardados en dicom_headers.csv)')
df = df.merge(df_headers, on='patientId', how='left')
antes_edad = len(df)
df = df[df['edad'].notna() & (df['edad'] >= EDAD_MINIMA) & (df['edad'] <= EDAD_MAXIMA)].copy()
print(f'#6. Filtro edad {EDAD_MINIMA}-{EDAD_MAXIMA} años: {antes_edad:,} -> {len(df):,} '
      f'(fuera {antes_edad - len(df):,} pediátricos/atípicos/sin edad)')

   - Leyendo headers de los DICOMs (tarda unos minutos, solo la primera vez)...
   - (headers guardados en dicom_headers.csv)
#6. Filtro edad 18-100 años: 14,863 -> 14,013 (fuera 850 pediátricos/atípicos/sin edad)


In [12]:
# Bloque 1.4 — Verificación final y guardado de la lista maestra
# Ni un patientId duplicado tras todo el filtro.
CSV_FILTRADO = CARPETA_DATASETS / 'dataset_filtrado.csv'  # lista maestra filtrada

n_dup = int(df['patientId'].duplicated().sum())
print(f'#7. Duplicados tras el filtro: {n_dup:,}')
assert n_dup == 0, 'Quedaron patientId duplicados'

# Se guarda la ruta del DICOM + dataset filtrado a CSV (lista maestra)
df['ruta_dicom'] = df['patientId'].apply(lambda pid: str(CARPETA_IMAGENES_TRAIN / f'{pid}.dcm'))
df_filtrado = df[['patientId', 'etiqueta', 'class', 'edad', 'sexo', 'vista', 'ruta_dicom']]
df_filtrado.to_csv(CSV_FILTRADO, index=False)
print(f'\nGuardado: {CSV_FILTRADO} ({len(df_filtrado):,} filas)')
print('Distribución final:')
for et, conteo in df_filtrado['etiqueta'].value_counts().items():
    nombre = 'con neumonía (1)' if et == 1 else 'sin neumonía (0)'
    print(f'   - {nombre}: {conteo:,} ({conteo / len(df_filtrado):.2%})')

#7. Duplicados tras el filtro: 0

Guardado: /content/drive/MyDrive/TESIS/DATASETS/dataset_filtrado.csv (14,013 filas)
Distribución final:
   - sin neumonía (0): 8,386 (59.84%)
   - con neumonía (1): 5,627 (40.16%)


## Bloque 2 — Fase 2: Transformación (4 funciones + demo)

Convierte cada DICOM en un array `3×224×224` normalizado, listo para DenseNet121. Aquí solo se definen las funciones y se prueban con 2 imágenes.

In [14]:
# Bloque 2.1 — Paso 1: DICOM -> array en escala de grises (0-255)
# DICOM (gris, 1024x1024) -> array RGB 3x224x224 normalizado (ver 2.5).
def leer_dicom_como_gris(ruta_dicom):
    """Lee un .dcm y devuelve array uint8 de 0-255 en escala de grises."""
    import pydicom
    ds = pydicom.dcmread(str(ruta_dicom))
    img = ds.pixel_array
    if str(ds.get('PhotometricInterpretation', '')) == 'MONOCHROME1':
        img = 255 - img  # fondo negro -> fondo blanco (estándar radiológico)
    img = img.astype(np.float32)
    if img.max() > 255:  # por si algún DICOM trae 16 bits
        img = img / img.max() * 255.0
    return np.clip(img, 0, 255).astype(np.uint8)

# Demo del paso 1 con una imagen real (no modifica nada en disco)
demo_dcm = sorted(CARPETA_IMAGENES_TRAIN.glob('*.dcm'))[0]
demo_gris = leer_dicom_como_gris(demo_dcm)
print(f'Paso 1 OK: {demo_dcm.name} -> forma={demo_gris.shape} dtype={demo_gris.dtype} '
      f'min={demo_gris.min()} max={demo_gris.max()}')

Paso 1 OK: 0004cfab-14fd-4e49-80ba-63a80b6bddd6.dcm -> forma=(1024, 1024) dtype=uint8 min=0 max=245


In [15]:
# Bloque 2.2 — Paso 2: achicar al tamaño que pide DenseNet121 (224x224)
def redimensionar(imagen_gris, tamano=224):
    """Achica la imagen al tamaño de entrada del modelo."""
    return np.array(Image.fromarray(imagen_gris).resize((tamano, tamano), Image.BILINEAR))

# Demo del paso 2 sobre el resultado del 2.1
demo_224 = redimensionar(demo_gris)
print(f'Paso 2 OK: {demo_gris.shape} -> {demo_224.shape}')

Paso 2 OK: (1024, 1024) -> (224, 224)


In [16]:
# Bloque 2.3 — Paso 3: gris -> 3 canales RGB (DenseNet121 espera 3 canales)
def gris_a_rgb(imagen_gris):
    """Copia el gris en 3 canales RGB (0-1)."""
    rgb = np.stack([imagen_gris, imagen_gris, imagen_gris], axis=0)
    return rgb.astype(np.float32) / 255.0

# Demo del paso 3 sobre el resultado del 2.2
demo_rgb = gris_a_rgb(demo_224)
print(f'Paso 3 OK: {demo_224.shape} -> forma={demo_rgb.shape} rango={demo_rgb.min():.2f}-{demo_rgb.max():.2f}')

Paso 3 OK: (224, 224) -> forma=(3, 224, 224) rango=0.00-0.95


In [17]:
# Bloque 2.4 — Paso 4: normalizar con media/desviación de ImageNet
# Media y desviación de ImageNet (las que espera DenseNet121 preentrenado).
# Se definen aquí porque solo se usan al normalizar.
MEDIA_IMAGENET = np.array([0.485, 0.456, 0.406], dtype=np.float32)
DESV_IMAGENET = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def normalizar_imagenet(imagen_rgb):
    """Ajusta los valores a la escala con la que se entrenó DenseNet121."""
    return (imagen_rgb - MEDIA_IMAGENET[:, None, None]) / DESV_IMAGENET[:, None, None]

# Demo del paso 4 sobre el resultado del 2.3
demo_norm = normalizar_imagenet(demo_rgb)
print(f'Paso 4 OK: forma={demo_norm.shape} min={demo_norm.min():.2f} max={demo_norm.max():.2f} media={demo_norm.mean():.2f}')

Paso 4 OK: forma=(3, 224, 224) min=-2.12 max=2.41 media=0.10


In [18]:
# Bloque 2.5 — Pipeline completo + demo con 2 imágenes (no modifica nada en disco)
def preprocesar_imagen(ruta_dicom, tamano=224):
    """Aplica los 4 pasos en orden: DICOM -> array (3, tamano, tamano) normalizado."""
    gris = leer_dicom_como_gris(ruta_dicom)
    gris_224 = redimensionar(gris, tamano)
    rgb = gris_a_rgb(gris_224)
    return normalizar_imagenet(rgb)


print('FASE 2 — demo con 2 imágenes de muestra')
for etiqueta in [1, 0]:
    pid = df_filtrado[df_filtrado['etiqueta'] == etiqueta].iloc[0]['patientId']
    arr = preprocesar_imagen(CARPETA_IMAGENES_TRAIN / f'{pid}.dcm')
    print(f'   - etiqueta={etiqueta}: forma={arr.shape} dtype={arr.dtype} '
          f'min={arr.min():.2f} max={arr.max():.2f} media={arr.mean():.2f}')
print('   (forma esperada: (3, 224, 224) lista para DenseNet121)')

FASE 2 — demo con 2 imágenes de muestra
   - etiqueta=1: forma=(3, 224, 224) dtype=float32 min=-2.08 max=1.70 media=-0.09
   - etiqueta=0: forma=(3, 224, 224) dtype=float32 min=-2.12 max=2.05 media=-0.00
   (forma esperada: (3, 224, 224) lista para DenseNet121)


## Bloque 3 — Fase 3: Particionamiento (split 70/15/15)

Reparte la lista maestra en `train` (70%), `val` (15%) y `test` (15%) conservando el mismo % de neumonía en los 3 grupos. Guarda los 3 CSV en `DATASETS/splits/`.

In [19]:
# Bloque 3 — FASE 3: PARTICIONAMIENTO (split estratificado 70/15/15 con numpy)
# Estratificado = cada split conserva el mismo % de neumonía que el total.
# No requiere sklearn: el reparto se hace con numpy (semilla fija = repetible).

SEMILLA = 42  # semilla fija: el reparto sale igual en cada corrida
CARPETA_SPLITS = CARPETA_DATASETS / 'splits'  # train.csv / val.csv / test.csv

def split_estratificado(indices_por_clase, proporciones=(0.70, 0.15, 0.15), semilla=SEMILLA):
    """Reparte índices por clase según proporciones. Devuelve 3 listas de índices."""
    rng = np.random.default_rng(semilla)
    partes = [[] for _ in proporciones]
    for _, idx in indices_por_clase:
        idx = np.array(idx)
        rng.shuffle(idx)
        n = len(idx)
        cortes = [int(n * proporciones[0]), int(n * (proporciones[0] + proporciones[1]))]
        partes[0].extend(idx[:cortes[0]])
        partes[1].extend(idx[cortes[0]:cortes[1]])
        partes[2].extend(idx[cortes[1]:])
    return partes


print('FASE 3 — PARTICIONAMIENTO (split estratificado 70/15/15)')
CARPETA_SPLITS.mkdir(exist_ok=True)
grupos = [(et, g.index.tolist()) for et, g in df_filtrado.groupby('etiqueta')]
idx_train, idx_val, idx_test = split_estratificado(grupos)
splits = {'train.csv': idx_train, 'val.csv': idx_val, 'test.csv': idx_test}
for nombre, idx in splits.items():
    sub = df_filtrado.loc[sorted(idx)]
    sub.to_csv(CARPETA_SPLITS / nombre, index=False)
    pct_pos = (sub['etiqueta'] == 1).mean()
    print(f'   - {nombre}: {len(sub):,} filas | neumonía={pct_pos:.2%}')
print(f'\nGuardados en: {CARPETA_SPLITS}')
print('Regla: el test/ solo se usa al final para reportar resultados.')

FASE 3 — PARTICIONAMIENTO (split estratificado 70/15/15)
   - train.csv: 9,808 filas | neumonía=40.15%
   - val.csv: 2,102 filas | neumonía=40.15%
   - test.csv: 2,103 filas | neumonía=40.18%

Guardados en: /content/drive/MyDrive/TESIS/DATASETS/splits
Regla: el test/ solo se usa al final para reportar resultados.


## Bloque 4 — Exportación (opcional, DESACTIVADO)

Aplica las 4 funciones a TODO el dataset y guarda los arrays ya listos en `DATASETS/dataset_filtrado/train|val|test/` como `.npy`. Ocupa ~8 GB: solo ejecútalo si vas a entrenar varias veces.

In [20]:
# Bloque 4 — EXPORTACIÓN (desactivada: quita el triple comillado para usarla)
CARPETA_SALIDA = CARPETA_DATASETS / 'dataset_filtrado'  # train/ val/ test/ con .npy

'''
"""
# Bloque 4 — EXPORTACIÓN (opcional, DESACTIVADA por defecto)
# Quita el triple comillado de abajo para activarla. Tarda bastante y ocupa ~8 GB.

def exportar_dataset_procesado():
    """Procesa todas las imágenes de cada split y las guarda como .npy."""
    from tqdm import tqdm  # barra de progreso (Colab ya lo trae instalado)
    for split in ['train', 'val', 'test']:
        df_split = pd.read_csv(CARPETA_SPLITS / f'{split}.csv')
        carpeta_split = CARPETA_SALIDA / split
        carpeta_split.mkdir(parents=True, exist_ok=True)
        print(f'   - Procesando {split}: {len(df_split):,} imágenes...')
        for _, fila in tqdm(df_split.iterrows(), total=len(df_split), desc=f'   - {split}'):
            arr = preprocesar_imagen(CARPETA_IMAGENES_TRAIN / f"{fila['patientId']}.dcm")
            np.save(carpeta_split / f"{fila['patientId']}.npy", arr)
        print(f'   - {split} OK: {len(df_split):,} archivos en {carpeta_split}')


# DESCOMENTA este bloque para aplicar la transformación a TODO el dataset
# y guardar los .npy en DATASETS/dataset_filtrado/train|val|test/ (~8 GB).
exportar_dataset_procesado()
print('EXPORTACIÓN COMPLETA')
'''

   - Procesando train: 9,808 imágenes...


   - train: 100%|██████████| 9808/9808 [07:58<00:00, 20.48it/s]


   - train OK: 9,808 archivos en /content/drive/MyDrive/TESIS/DATASETS/dataset_filtrado/train
   - Procesando val: 2,102 imágenes...


   - val: 100%|██████████| 2102/2102 [01:23<00:00, 25.16it/s]


   - val OK: 2,102 archivos en /content/drive/MyDrive/TESIS/DATASETS/dataset_filtrado/val
   - Procesando test: 2,103 imágenes...


   - test: 100%|██████████| 2103/2103 [01:23<00:00, 25.29it/s]

   - test OK: 2,103 archivos en /content/drive/MyDrive/TESIS/DATASETS/dataset_filtrado/test
EXPORTACIÓN COMPLETA


## Bloque 5 — Verificación (ubicación y conteo, solo lectura)

Comprueba que `DATASETS/dataset_filtrado/train|val|test/` existan, que cada carpeta tenga tantos `.npy` como filas su CSV y cuánto pesan. No modifica nada.

In [23]:
# Bloque 5 — VERIFICACIÓN: ubicación, conteo y tamaño (solo lectura)
# Compara lo que DICEN los CSV contra lo que REALMENTE hay en disco.
CARPETA_SALIDA = CARPETA_DATASETS / 'dataset_filtrado'  # donde el Bloque 4 guarda los .npy

print('=' * 70)
print('BLOQUE 5 — VERIFICACIÓN del dataset exportado')
print('=' * 70)
print(f'\nCarpeta padre: {CARPETA_SALIDA}')
print(f'Contenido: {sorted([p.name for p in CARPETA_SALIDA.iterdir()])}')
rng = np.random.default_rng(42)
ok = True
for split in ['train', 'val', 'test']:
    en_csv = len(pd.read_csv(CARPETA_SPLITS / f'{split}.csv'))
    archivos = sorted((CARPETA_SALIDA / split).glob('*.npy'))
    en_disco = len(archivos)
    estado = 'OK' if en_csv == en_disco else 'FALTAN'
    if en_csv != en_disco:
        ok = False
    print(f'   - {split}/: en CSV={en_csv:,} | en disco={en_disco:,} .npy -> {estado}')
    if archivos:
        muestra = rng.choice(archivos, size=min(3, len(archivos)), replace=False)
        for f in muestra:
            arr = np.load(f)  # lectura real del archivo
            if arr.shape != (3, 224, 224) or arr.dtype != np.float32:
                ok = False
                print(f'      ! Archivo malo: {f.name} forma={arr.shape} tipo={arr.dtype}')
            else:
                print(f'      Lectura real OK: {f.name} forma=(3, 224, 224) float32')
total_gb = sum(p.stat().st_size for p in CARPETA_SALIDA.rglob('*.npy')) / 1e9
print(f'\nTamaño total: {total_gb:.2f} GB')
print('VERIFICACIÓN COMPLETA: todo OK' if ok else 'HAY PROBLEMAS: re-ejecuta el Bloque 4')

BLOQUE 5 — VERIFICACIÓN del dataset exportado

Carpeta padre: /content/drive/MyDrive/TESIS/DATASETS/dataset_filtrado
Contenido: ['test', 'train', 'val']
   - train/: en CSV=9,808 | en disco=9,808 .npy -> OK
      Lectura real OK: ad750a01-7b7d-43b5-b4b2-42bf12a603bd.npy forma=(3, 224, 224) float32
      Lectura real OK: 1caf3708-4998-4594-bf7f-ea4beb24b6f5.npy forma=(3, 224, 224) float32
      Lectura real OK: c1dfd251-ba52-4ecb-94fa-f98ce08a07d5.npy forma=(3, 224, 224) float32
   - val/: en CSV=2,102 | en disco=2,102 .npy -> OK
      Lectura real OK: 1847b55c-2fed-494c-8d76-7f76630ba39f.npy forma=(3, 224, 224) float32
      Lectura real OK: b23184fc-0e6e-4802-a451-da82e2ac6e9a.npy forma=(3, 224, 224) float32
      Lectura real OK: d8155412-2944-43e5-ac3c-64ec82f1cbd2.npy forma=(3, 224, 224) float32
   - test/: en CSV=2,103 | en disco=2,103 .npy -> OK
      Lectura real OK: 91cc9818-4778-4659-978c-a5a472e3c416.npy forma=(3, 224, 224) float32
      Lectura real OK: f9c4f02b-52b5-4f36-96